## START

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# -- takes a few mins to run first time - sentence trainsformers is about 5GB in size...
# also hugging face complains - I need HF_TOKEN to enable higher rate limits and faster downloads.

# The first time I run this, it downloads the model (~80 MB) and the tokenizer from HuggingFace. 
# The tokenizer turns text into something the model can read. After that, both load from a local cache.

# Trying it with simple examples
# Let's see how embeddings work on a few examples.

# We'll start with a query:

q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
# v1 # prints embedding vector like array([ 2.13903598e-02, -7.39799812e-02,  1.42071780e-03,  2.13816296e-02, ...

# v1 is a vector, an array of 384 numbers. Each number stands for some concept the model learned. 
# We can't read off what any one of them means. But two vectors with similar values point to texts about similar things.


In [3]:
# Encode our simplest document:

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
# Next, we compare the query against the document using dot product:
# this is cosine similarity metrics - dist between embedding vectors - but we use matrix multiplication for simplicity here :

v1.dot(dv)

# We get 0.32. -- np.float32(0.32332408)

np.float32(0.32332408)

In [5]:
# Now we try an unrelated query:

q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

# This time the similarity with the document should be much smaller:

v2.dot(dv)

# And we get 0.01. -- np.float32(0.019730505) - very small because Q and A are symantically dissimilar

np.float32(0.019730505)

In [6]:
q2 = "How much is the fish?"
v2 = model.encode(q2)

# totally unrelated Q - the similarity with the document should be smaller:

v2.dot(dv)

np.float32(-0.073930666)

In [7]:
# EXPLANATION:

# The first score for q1 vs d (0.32) is higher, so that query is more similar to the document about registration. 
# The second score for q2 vs d sits near 0, because installing Docker has nothing to do with registration. 
# A score near 0 means the two vectors are about as different as they can be.

# That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

# Cosine similarity
# The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. When both vectors are normalized, 
# the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

# Cosine similarity measures the angle between two vectors, ignoring their length:

# 1.0 = same direction (similar)
# 0.0 = perpendicular (unrelated)
# -1.0 = opposite direction (opposite meaning)
# Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

# cos(0) = 1 - vectors point in the same direction
# cos(90) = 0 - vectors are perpendicular
# cos(180) = -1 - vectors point in opposite directions
# Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use v1.dot(dv) to compare texts.

# In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space 
# where most vectors have positive components. There's no concept of "opposite meaning" that maps to a vector pointing the other way.

## Embedding Our Dataset

In [8]:
# Loading the data
# In module 1 we created ingest.py for loading the FAQ data.

# Download it into your project:
# choco install wget
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
# We use it here:

from ingest import load_faq_data

documents = load_faq_data()

In [9]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [10]:
len(documents)

1350

In [11]:
# Generating embeddings
# Each document is a Python dictionary with a question and an answer. We embed both together. That way a query can match against the question text and the answer text in our index.

# Build one text per document:

texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

texts[10] # we combined Q and A - see documents[10] from above

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [12]:
# Now we generate the embeddings. We have about 1200 texts here. We won't hand the model all of them at once. 
# That takes a long time, and we can't see what's happening inside. Instead we split them into batches.

# First we import tqdm so we can watch the progress:
# uv add tqdm

from tqdm.auto import tqdm
# Next we chunk the dataset into batches of 50 and encode each batch:

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)
# We end up with 1350 vectors. On a GPU this is fast. Most of us run on Codespaces without a GPU, so it takes a bit, but it's a one-off.


  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [13]:
# We turn them into a 2-dimensional array (matrix) where
# rows are documents (vectors)
# columns are dimensions of the vectors

import numpy as np
X = np.array(vectors)
X.shape

# Calling X.shape returns (1350, 384) - number of documents vs number of dimensions.

(1350, 384)

## Vector search

In [14]:
# Scoring documents
# We have a matrix X with all document embeddings. We take a query, compare it against every document, and keep the most similar ones.

# When a query comes in, we embed it:

query = "Can I still join the course after the start date?"
v_query = model.encode(query)
# Next, we compute the dot product against all documents:

scores = X.dot(v_query)
# This is matrix-vector multiplication. Each element i of scores is the cosine similarity between document i (row i of X) and v_query.

In [16]:
# Best match
# The highest score is the most similar document:

idx = np.argmax(scores)
idx, scores[idx]
# This returns document 2 with score 0.76.

# The index and score may differ for you. Our FAQ is a living document, so we add and remove entries over time.



(np.int64(2), np.float32(0.7629411))

In [17]:
# Let's see which document it is:

documents[idx]
# We see:

# {"id": "3f1424af17",
#  "course": "data-engineering-zoomcamp",
#  "section": "General Course-Related Questions",
#  "question": "Course: Can I still join the course after the start date?",
#  "answer": "Yes, even if you don't register, you're still eligible..."}

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [18]:
# Top 5 results
# Usually we want more than the single best match, so let's pull the top 5.

# np.argsort sorts from lowest to highest, so the last 5 are the top ones:

top5 = np.argsort(scores)[-5:]

In [25]:
# They come out smallest-first, so we reverse them to get the highest first:

top5 = top5[::-1]
top5

array([  7, 538, 907, 625,   2])

In [20]:
# Now we can read off the top 5 scores:

scores[top5]

array([0.7629411 , 0.7579373 , 0.71921325, 0.6536313 , 0.56009996],
      dtype=float32)

In [26]:
# There's a shorter trick I usually reach for. We negate the scores first, so the largest becomes the smallest. 
# Then argsort puts the best matches at the front.

# Here it is in one line:

top5 = np.argsort(-scores)[:5]
# It looks cryptic the first time you see it. But it's a common way to turn a min-sort into a max-sort.

# Let's read off the actual documents:

for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()
    
# This is vector search in its simplest form. We embed the query, compute dot products against all documents, and return the highest-scoring ones.

# We return 5 and not the single best for a reason. The answer to a question can be spread across several documents. 
# One holds part of it, another fills in the rest. Sometimes the top result isn't the right one but the second is. 
# We send all 5 to the LLM and let it combine them.

# The number 5 is a starting point, picked on gut feeling. Later, when we evaluate search quality, we can test 
# whether 3 or 10 works better for our data.

# Doing this by hand with numpy is fine for a small dataset. A larger one needs a library that also handles filtering and ranking. 
# That's what we turn to next.

0.7629411
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579373
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.71921325
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'

## Vector Search with minsearch